In [ ]:
from pyspark.sql import SparkSession

try:
    spark = SparkSession.active()
except RuntimeError:
    spark = SparkSession.builder.getOrCreate()

In [ ]:
CATALOG_NAME = "kelp_catalog"

In [ ]:
# Cleanup
spark.sql(f"DROP VOLUME IF EXISTS {CATALOG_NAME}.kelp_bronze.landing_volume")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG_NAME}.kelp_bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG_NAME}.kelp_silver CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG_NAME}.kelp_gold CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG_NAME}.kelp_sdp_monitoring CASCADE")
spark.sql(f"DROP CATALOG IF EXISTS {CATALOG_NAME} CASCADE")

In [ ]:

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")

spark.sql(f"Create Schema if not exists {CATALOG_NAME}.kelp_bronze")
spark.sql(f"Create Schema if not exists {CATALOG_NAME}.kelp_silver")
spark.sql(f"Create Schema if not exists {CATALOG_NAME}.kelp_gold")
spark.sql(f"Create Schema if not exists {CATALOG_NAME}.kelp_quarantine")
spark.sql(f"Create Schema if not exists {CATALOG_NAME}.kelp_sdp_monitoring")
spark.sql(f"Create Volume if not exists {CATALOG_NAME}.kelp_bronze.landing_volume")


In [ ]:
from faker import Faker
import numpy as np

fake = Faker()

num_customers = 100
num_customers_bad = 10
num_orders = 200
num_orders_bad = 20
customers = []

countries = [
    "United States",
    "Canada",
    "United Kingdom",
    "Australia",
    "Germany",
    "France",
    "India",
    "China",
    "Brazil",
    "Mexico",
]
products = [
    "Laptop",
    "Smartphone",
    "Headphones",
    "Camera",
    "Smartwatch",
    "Tablet",
    "Printer",
    "Monitor",
    "Keyboard",
    "Mouse",
]
store = ["Webshop", "Retail Store", "Mobile App", "Phone Order"]
order_states = ["Pending", "Shipped", "Delivered", "Cancelled"]
customer_ids = [fake.uuid4() for _ in range(num_customers)]
for i in customer_ids:
    customers.append(
        {
            "user_id": i,
            "first_name": fake.first_name(),
            "last_name": fake.last_name(),
            "country": np.random.choice(countries).item(),
        }
    )

for i in range(num_customers_bad):
    customers.append(
        {
            "user_id": fake.uuid4(),
            "first_name": fake.first_name(),
            "last_name": fake.last_name(),
            "country": "Unknown",
        }
    )


orders = []
for i in range(num_orders):
    orders.append(
        {
            "order_id": fake.uuid4(),
            "user_id": np.random.choice(customer_ids).item(),
            "product": np.random.choice(products).item(),
            "quantity": np.random.randint(1, 10),
            "store": np.random.choice(store).item(),
            "price": round(np.random.uniform(10.0, 100.0), 2),
            "order_state": np.random.choice(order_states).item(),
        }
    )

for i in range(num_orders_bad):
    orders.append(
        {
            "order_id": fake.uuid4(),
            "user_id": np.random.choice(customer_ids).item(),
            "product": np.random.choice(products).item(),
            "quantity": np.random.randint(-10, -1),
            "store": np.random.choice(store).item(),
            "price": round(np.random.uniform(10.0, 100.0), 2),
            "order_state": "Thrown Away",
        }
    )

customers_df = spark.createDataFrame(customers)
customers_df.printSchema()
orders_df = spark.createDataFrame(orders)
orders_df.printSchema()

customers_df.write.mode("overwrite").parquet(
    f"dbfs:/Volumes/{CATALOG_NAME}/kelp_bronze/landing_volume/customers/"
)
orders_df.write.format("parquet").mode("overwrite").save(
    f"dbfs:/Volumes/{CATALOG_NAME}/kelp_bronze/landing_volume/orders/"
)